In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
from google.colab import files
uploaded=files.upload

In [3]:
spark = (
    SparkSession.builder
    .appName("ExpenseMonitoringSystem")
    .getOrCreate()
)

In [4]:
df = spark.read.csv(
    "expenses.csv",
    header=True,
    inferSchema=True
)

In [5]:
df = df.withColumn(
    "date",
    to_date("date")
)

In [6]:
df = df.withColumn(
    "month",
    date_format("date", "yyyy-MM")
)

In [11]:
monthly_spending = (
    df.groupBy("user_id", "month")
    .agg(
        sum("amount")
        .alias("total_spent")
    )
)
monthly_spending.show()

+-------+-------+-----------+
|user_id|  month|total_spent|
+-------+-------+-----------+
|      3|2026-06|       4900|
|      3|2026-07|       3000|
|      1|2026-07|       4320|
|      1|2026-06|       3600|
|      2|2026-06|       3500|
|      2|2026-07|       6900|
+-------+-------+-----------+



In [12]:
unusual_expenses = (
    df.filter(
        col("amount") > 1000
    )
)
unusual_expenses.show()

+-------+-------------+----------+------+-------------------+-------+
|user_id|     category|      date|amount|        description|  month|
+-------+-------------+----------+------+-------------------+-------+
|      2|     Shopping|2026-06-03|  2500|   Clothes Purchase|2026-06|
|      1|     Shopping|2026-06-05|  1200|         Headphones|2026-06|
|      1|    Utilities|2026-06-08|  1800|   Electricity Bill|2026-06|
|      3|     Shopping|2026-06-10|  3500| Laptop Accessories|2026-06|
|      2|    Utilities|2026-07-02|  1600|      Internet Bill|2026-07|
|      2|     Shopping|2026-07-05|  4200|       Mobile Phone|2026-07|
|      2|Entertainment|2026-07-08|  1100|     Concert Ticket|2026-07|
|      3|    Utilities|2026-07-09|  2100|Water & Electricity|2026-07|
|      1|     Shopping|2026-07-10|  2700|          Furniture|2026-07|
+-------+-------------+----------+------+-------------------+-------+



In [13]:
unusual_expenses.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("unusual_expenses.csv")